In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

In [2]:
models_dir = 'models'
data_dir = 'data'

Path(models_dir).mkdir(exist_ok=True, parents=True)
Path(data_dir).mkdir(exist_ok=True, parents=True)

In [3]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Сохраним эталонные данные для Evidently (без заголовков, как в CSV)
ref_df = pd.DataFrame(X_train, columns=iris.feature_names)
ref_df["target"] = y_train
ref_df.to_csv(f"{data_dir}/reference.csv", index=False)

In [5]:
rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_train, y_train)
joblib.dump(rf, f"{models_dir}/iris_rf.pkl")

['models/iris_rf.pkl']

In [6]:
cat = CatBoostClassifier(iterations=20, depth=3, verbose=True, random_seed=42)
cat.fit(X_train, y_train)
cat.save_model("models/iris_catboost.cbm")
print("CatBoost model saved.")

Learning rate set to 0.5
0:	learn: 0.6176152	total: 46.1ms	remaining: 877ms
1:	learn: 0.4050792	total: 46.3ms	remaining: 417ms
2:	learn: 0.2942950	total: 46.4ms	remaining: 263ms
3:	learn: 0.2391299	total: 46.6ms	remaining: 186ms
4:	learn: 0.1945792	total: 46.7ms	remaining: 140ms
5:	learn: 0.1792512	total: 46.8ms	remaining: 109ms
6:	learn: 0.1767703	total: 46.9ms	remaining: 87.1ms
7:	learn: 0.1611172	total: 47ms	remaining: 70.6ms
8:	learn: 0.1431223	total: 47.2ms	remaining: 57.7ms
9:	learn: 0.1277378	total: 47.3ms	remaining: 47.3ms
10:	learn: 0.1177199	total: 47.4ms	remaining: 38.8ms
11:	learn: 0.1030741	total: 47.6ms	remaining: 31.7ms
12:	learn: 0.0951777	total: 48ms	remaining: 25.9ms
13:	learn: 0.0896060	total: 48.2ms	remaining: 20.7ms
14:	learn: 0.0809665	total: 48.4ms	remaining: 16.1ms
15:	learn: 0.0754992	total: 48.5ms	remaining: 12.1ms
16:	learn: 0.0684031	total: 48.7ms	remaining: 8.59ms
17:	learn: 0.0635795	total: 48.9ms	remaining: 5.43ms
18:	learn: 0.0605153	total: 49ms	remainin

In [7]:
class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.net(x)


In [8]:
train_data = torch.tensor(X_train, dtype=torch.float32)
train_labels = torch.tensor(y_train, dtype=torch.long)

In [9]:
model = IrisNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-2)
model.train()


IrisNet(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=3, bias=True)
  )
)

In [10]:
for epoch in range(200):
    optimizer.zero_grad()
    output = model(train_data)
    loss = criterion(output, train_labels)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.6f}")

torch.save(model.state_dict(), "models/iris_nn.pt")
print("PyTorch model saved.")


Epoch   0 | Loss: 1.284937
Epoch  20 | Loss: 0.765340
Epoch  40 | Loss: 0.287241
Epoch  60 | Loss: 0.094761
Epoch  80 | Loss: 0.066024
Epoch 100 | Loss: 0.060361
Epoch 120 | Loss: 0.057761
Epoch 140 | Loss: 0.055990
Epoch 160 | Loss: 0.054591
Epoch 180 | Loss: 0.053412
PyTorch model saved.


In [11]:
model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    nn_preds = model(X_test_t).argmax(dim=1).numpy()

rf_preds = rf.predict(X_test)
cat_preds = cat.predict(X_test)

print(f"Random Forest:  {accuracy_score(y_test, rf_preds):.4f}")
print(f"CatBoost:       {accuracy_score(y_test, cat_preds):.4f}")
print(f"IrisNet:        {accuracy_score(y_test, nn_preds):.4f}")


Random Forest:  1.0000
CatBoost:       1.0000
IrisNet:        1.0000
